### The Baselines

* **Random Classifier:**
* **Heuristic:**
* **Multi-Class Linear Classifier:** 

### Neural Network

* **Basic Neural Network / Multilayer Perceptron:**
* **Regularized NN:**

### Natural Language Processing:

* **Bag of Words Model:**
* **RNN Sentiment Analysis:**

### Extra

* **Attention Models:** Use attention mechanisms on the titles/descriptions
* **Some Fancy Pretrained Model maybe BERT too**
* **Just run it into an LLM and see how it does**


## Load and Process Data

In [1]:
# access parquet :D files with pandas
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from collections import Counter
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

df = pl.read_parquet('data/youtube_data.parquet')
df

title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,like_count,comment_count,description,video_id,channel_id,video_tags,kind,publish_date,langauge,like_tier
str,cat,u8,i16,i16,"datetime[μs, UTC]",cat,i64,i32,i32,str,str,str,str,cat,"datetime[μs, UTC]",cat,cat
"""Nyasha David - Tsvodi (Officia…","""Nyasha David""",1,0,15,2026-02-18 00:00:00 UTC,"""ZW""",430573,13860,1139,"""#Tsvodi #nyashadavid #mafindif…","""07pinMbPq2Q""","""UCx1LPxEtJWIpvtXTYadAdKg""","""""","""youtube#video""",2026-02-09 00:00:00 UTC,"""und""","""pretty_viral"""
"""WAR MACHINE | Official Trailer…","""Netflix""",2,0,0,2026-02-18 00:00:00 UTC,"""ZW""",12531660,191591,15336,"""During the final stage of U.S.…","""AFuE1LRxm80""","""UCWOA1ZGywLbqmigxE4Qlvuw""","""Alan Ritchson, Alan Ritchson W…","""youtube#video""",2026-02-04 00:00:00 UTC,"""en-US""","""really_viral"""
"""10 Seconds vs 1 Hour MODERN MO…","""Cash""",3,0,47,2026-02-18 00:00:00 UTC,"""ZW""",491455,6139,1656,"""Instagram: https://www.instagr…","""EbBjlznDwzk""","""UC0eLBYhxW9HC0P9PXQ73mpQ""","""Cash, Nico, Nico and Cash, Cas…","""youtube#video""",2026-02-16 00:00:00 UTC,"""en-US""","""a_little_viral"""
"""Master H - Impossible (Officia…","""Master H""",4,0,3,2026-02-18 00:00:00 UTC,"""ZW""",532388,14552,1281,"""Artist : Master H Song : Impos…","""TIG5IzusQ24""","""UC5jrU8WxzE16gPTnlqQAqew""","""""","""youtube#video""",2026-02-06 00:00:00 UTC,"""en""","""pretty_viral"""
"""“Spider-Noir” – Authentic Blac…","""Prime Video""",5,0,45,2026-02-18 00:00:00 UTC,"""ZW""",13406903,239798,10614,"""With no power comes no respons…","""HgMbkitzhEM""","""UCQJWtTnAHhEG5w4uN0udnUQ""","""spider noir official teaser, s…","""youtube#video""",2026-02-12 00:00:00 UTC,"""en""","""really_viral"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""انهيار الطبيبة هديل الجمل عند …","""شبكة قدس الاخبارية""",46,4,4,2023-10-26 00:00:00 UTC,"""AE""",1064501,8823,815,"""تابعوا #شبكة_قدس عبر المنصات ا…","""OcPejiPT07U""","""UCFFuMGTQXLitNuiU4SIe5vQ""","""فلسطين, القدس, غزة, تاريخ فلسط…","""youtube#video""",2023-10-22 15:04:53 UTC,"""ar""","""a_little_viral"""
"""بدهم ايانا نترك بيتنا !""","""وليد ونور | Waleed & Noor""",47,3,3,2023-10-26 00:00:00 UTC,"""AE""",413648,36218,2070,"""حساباتنا على مواقع التواصل الإ…","""qX9XD-5ZGIE""","""UCHywC_HXMWAM6-XvjY8gnUw""","""وليد ونور, نور غسان, وليدمقداد…","""youtube#video""",2023-10-22 16:10:45 UTC,"""ar""","""pretty_viral"""
"""""إسرائيل"" تنزف!.. هل ينهار الا…","""المخبر الاقتصادي - Mokhbir Eqt…",48,2,2,2023-10-26 00:00:00 UTC,"""AE""",2076478,197026,12930,"""لو انت شركة أو تاجر اتواصل مع …","""s7n2wzIWjtY""","""UC4kRorAXuIkyIX6vwXKaLWg""","""المخبر الاقتصادي, اخبار الاقتص…","""youtube#video""",2023-10-19 17:00:08 UTC,"""ar""","""really_viral"""


In [2]:
# Dedpulicate videos
df = df.drop("like_count")
df = df.sort("snapshot_date")
df = df.unique(subset=["video_id"], keep="last")

# Sort the new unique dataframe by when the videos were published
df = df.sort("publish_date")

# Split by cutoff date
cutoff_date = df['publish_date'].quantile(0.8, "lower")
print(f"Using publish_date {cutoff_date} as the cutoff point.\n")

train_df = df.filter(pl.col("publish_date") <= cutoff_date)
test_df = df.filter(pl.col("publish_date") > cutoff_date)

X_train = train_df.drop("like_tier")
y_train = train_df.get_column("like_tier")

X_test = test_df.drop("like_tier")
y_test = test_df.get_column("like_tier")

X_train_pd = X_train.to_pandas()    
X_test_pd = X_test.to_pandas()

print(f"Training set size: {len(X_train)} unique videos (past)")
print(f"Testing set size:  {len(X_test)} unique videos (future)")

Using publish_date 2025-11-05 00:00:00+00:00 as the cutoff point.

Training set size: 366469 unique videos (past)
Testing set size:  90674 unique videos (future)


In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


#### Random Classifier

In [7]:
# a random classifier that takes account in class distributions
# Calculate class probabilities directly from the y_train Series
counts_df = y_train.value_counts()
labels = counts_df.get_column("like_tier").to_list()
probs = (counts_df.get_column("count") / len(y_train)).to_list()

random_preds = np.random.choice(
    labels,
    size=len(y_test),
    p=probs
)

print("Random Classifier Evaluation Report:")
print(classification_report(y_test, random_preds, zero_division=0))

Random Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      0.49      0.61     73784
  pretty_viral       0.16      0.32      0.21     14234
  really_viral       0.03      0.13      0.04      2486
   super_viral       0.00      0.06      0.00       170

      accuracy                           0.46     90674
     macro avg       0.25      0.25      0.22     90674
  weighted avg       0.69      0.46      0.53     90674



### Huristic Classifiers

#### Majority Class

In [ ]:
# find the most common class in the training labels
majority_class = y_train.mode()[0]

# predict that class for every test sample
heuristic_preds = [majority_class] * len(y_test)


print("Majority Class:", majority_class)
print("Huristic Classifier Evaluation Report (Majority Class):")
print(classification_report(y_test, heuristic_preds, zero_division=0))

Majority Class: a_little_viral
Huristic Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      1.00      0.90     73784
  pretty_viral       0.00      0.00      0.00     14234
  really_viral       0.00      0.00      0.00      2486
   super_viral       0.00      0.00      0.00       170

      accuracy                           0.81     90674
     macro avg       0.20      0.25      0.22     90674
  weighted avg       0.66      0.81      0.73     90674



In [ ]:
heuristic_feature = "view_count" 
tier_labels = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]

q1_val = 0.70
q2_val = 0.88
q3_val = 0.97

q1 = X_train.get_column(heuristic_feature).quantile(q1_val)
q2 = X_train.get_column(heuristic_feature).quantile(q2_val)
q3 = X_train.get_column(heuristic_feature).quantile(q3_val)

print(f"Predict '{tier_labels[0]}': <= {q1:.0f}")
print(f"Predict '{tier_labels[1]}': > {q1:.0f} and <= {q2:.0f}")
print(f"Predict '{tier_labels[2]}': > {q2:.0f} and <= {q3:.0f}")
print(f"Predict '{tier_labels[3]}': > {q3:.0f}\n")

heuristic_preds_series = X_test.select(
    pl.when(pl.col(heuristic_feature) <= q1).then(pl.lit(tier_labels[0]))
    .when(pl.col(heuristic_feature) <= q2).then(pl.lit(tier_labels[1]))
    .when(pl.col(heuristic_feature) <= q3).then(pl.lit(tier_labels[2]))
    .otherwise(pl.lit(tier_labels[3]))
    .alias("predictions")
).get_column("predictions")

y_test_list = y_test.to_list()
heuristic_preds = heuristic_preds_series.to_list()

print("Heuristic Classifier Evaluation Report (View Count Quantiles):")
print(classification_report(y_test_list, heuristic_preds, zero_division=0))

--- Heuristic Thresholds ---
Predict 'a_little_viral': <= 711368
Predict 'pretty_viral': > 711368 and <= 2470013
Predict 'really_viral': > 2470013 and <= 13059227
Predict 'super_viral': > 13059227

Heuristic Classifier Evaluation Report (View Count Quantiles):
                precision    recall  f1-score   support

a_little_viral       0.86      0.98      0.92     73784
  pretty_viral       0.51      0.21      0.30     14234
  really_viral       0.58      0.25      0.35      2486
   super_viral       0.57      0.50      0.53       170

      accuracy                           0.84     90674
     macro avg       0.63      0.49      0.52     90674
  weighted avg       0.80      0.84      0.80     90674



### Linear Classifiers

#### Torch model
* 50 epochs
* Numeric Features
* SGD optimizer

In [ ]:
EPOCHS = 10

# numeric only first not like_count lmao
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# Convert everything to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

# Create Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

# Simple linear classifier
model = nn.Linear(len(numeric_features), len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
# We'll use SGD to make the baseline less powerful Adam does too much for us
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Training Linear Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    
    # Move predictions back to CPU for Scikit-Learn
    predicted_np = predicted.cpu().numpy()

print("\nLinear Classifier Numeric Only Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

Training Linear Classifier...

Linear Classifier Numeric Only Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.88      0.99      0.93     73784
  pretty_viral       0.66      0.29      0.40     14234
  really_viral       0.72      0.33      0.45      2486
   super_viral       0.61      0.55      0.58       170

      accuracy                           0.86     90674
     macro avg       0.72      0.54      0.59     90674
  weighted avg       0.84      0.86      0.83     90674



#### Scikit LogisticRegression model
* 100 max_iter
* Numeric Features
* LBFGS solver

In [10]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

print("Training Scikit-Learn Linear Classifier...")

model = LogisticRegression(
    solver='lbfgs',      # Standard solver for multi-class
    max_iter=100,       # Gives it enough time to converge
    random_state=42      # Keeps results reproducible
)
model.fit(X_train_scaled, y_train_np)
predicted_np = model.predict(X_test_scaled)

print("\nScikit-Learn Linear Classifier Evaluation Report:")

target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_np, predicted_np, labels=target_names, zero_division=0))

Training Scikit-Learn Linear Classifier...

Scikit-Learn Linear Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.91      0.97      0.94     73784
  pretty_viral       0.67      0.45      0.54     14234
  really_viral       0.68      0.41      0.51      2486
   super_viral       0.58      0.65      0.61       170

      accuracy                           0.88     90674
     macro avg       0.71      0.62      0.65     90674
  weighted avg       0.86      0.88      0.86     90674



### Multilayer Perceptron 
* 50 Epochs
* Numerical Features
* Adam Optimizer


In [11]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)


model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension: 5 features
Training MLP Classifier...

MLP Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.94      0.94     73784
  pretty_viral       0.63      0.62      0.62     14234
  really_viral       0.61      0.62      0.61      2486
   super_viral       0.51      0.79      0.62       170

      accuracy                           0.88     90674
     macro avg       0.67      0.74      0.70     90674
  weighted avg       0.88      0.88      0.88     90674



### Multilayer Perceptron 
* 50 Epochs
* Numerical + Categorical Features
* Adam Optimizer


In [12]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Numeric + Categorical Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension: 303 features
Training MLP Classifier...

MLP Classifier Numeric + Categorical Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.96      0.95     73784
  pretty_viral       0.68      0.63      0.66     14234
  really_viral       0.69      0.53      0.60      2486
   super_viral       0.50      0.75      0.60       170

      accuracy                           0.89     90674
     macro avg       0.70      0.72      0.70     90674
  weighted avg       0.89      0.89      0.89     90674



### Natural Language Processing

#### Bag of Words + Logistic Regression


In [13]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

# combine title and description into a single text feature
X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        # using frequency BoW
        ('bow', CountVectorizer(max_features=2000), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with BoW: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# count examples per class
class_counts = np.bincount(y_train_mapped)
print("Class counts:", class_counts)

# inverse frequency weights to handle class imbalance
class_weights = {i: max(class_counts)/count for i, count in enumerate(class_counts)}
print("Class weights:", class_weights)

clf = LogisticRegression(
    solver='saga',
    max_iter=3000,
    class_weight=class_weights
)

clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with BoW Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

New Input Dimension with BoW: 2303 features
Class counts: [180600 117454  51222  17193]
Class weights: {0: np.float64(1.0), 1: np.float64(1.5376232397364074), 2: np.float64(3.525828745460935), 3: np.float64(10.50427499563776)}

Logistic Regression with BoW Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.93      0.93     73784
  pretty_viral       0.60      0.59      0.59     14234
  really_viral       0.53      0.70      0.60      2486
   super_viral       0.36      0.82      0.50       170

      accuracy                           0.87     90674
     macro avg       0.60      0.76      0.66     90674
  weighted avg       0.87      0.87      0.87     90674



### MLP (Numeric + Categorical + BoW)

In [14]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('bow', CountVectorizer(max_features=2000), "text")
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with BoW: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


print("Training MLP with Bag of Words...")
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier (Numeric + Categorical + BoW) Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension with BoW: 2303 features
Training MLP with Bag of Words...

MLP Classifier (Numeric + Categorical + BoW) Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.96      0.93      0.94     73784
  pretty_viral       0.64      0.74      0.68     14234
  really_viral       0.71      0.74      0.72      2486
   super_viral       0.65      0.70      0.67       170

      accuracy                           0.89     90674
     macro avg       0.74      0.78      0.76     90674
  weighted avg       0.90      0.89      0.89     90674



#### TF-IDF + Logistic Regression

In [15]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]
X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('tfidf', TfidfVectorizer(max_features=2000, sublinear_tf=True), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with TF-IDF: {input_dim} features")

# count examples per class
class_counts = np.bincount(y_train_mapped)
print("Class counts:", class_counts)

# inverse frequency weights to handle class imbalance
class_weights = {i: max(class_counts)/count for i, count in enumerate(class_counts)}
print("Class weights:", class_weights)

clf = LogisticRegression(
    solver='saga',
    max_iter=3000,
    class_weight=class_weights
)

clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with TF-IDF Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

New Input Dimension with TF-IDF: 2303 features
Class counts: [180600 117454  51222  17193]
Class weights: {0: np.float64(1.0), 1: np.float64(1.5376232397364074), 2: np.float64(3.525828745460935), 3: np.float64(10.50427499563776)}

Logistic Regression with TF-IDF Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.93      0.93     73784
  pretty_viral       0.61      0.61      0.61     14234
  really_viral       0.55      0.71      0.62      2486
   super_viral       0.42      0.86      0.56       170

      accuracy                           0.87     90674
     macro avg       0.63      0.78      0.68     90674
  weighted avg       0.88      0.87      0.87     90674



### MLP (Numeric + Categorical + TF-IDF)

In [16]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('tfidf', TfidfVectorizer(max_features=2000, sublinear_tf=True), "text")
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with TF-IDF: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


print("Training MLP with TFIDF...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier (Numeric + Categorical + TF-IDF) Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension with TF-IDF: 2303 features
Training MLP with TFIDF...

MLP Classifier (Numeric + Categorical + TF-IDF) Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.96      0.94      0.95     73784
  pretty_viral       0.70      0.78      0.74     14234
  really_viral       0.74      0.71      0.73      2486
   super_viral       0.64      0.75      0.69       170

      accuracy                           0.91     90674
     macro avg       0.76      0.80      0.78     90674
  weighted avg       0.92      0.91      0.91     90674



#### RNN Sentiment Analysis

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def label_sentiment(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score > 0.05:
        # positive
        return 1
    elif score < -0.05:
        # negative
        return -1
    else:
        # neutral
        return 0

df = df.to_pandas()

# apply VADER
df["sentiment"] = (df["title"].astype(str) + " " + df["description"].astype(str)).apply(label_sentiment)

df.head()

,title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,comment_count,description,video_id,channel_id,video_tags,kind,publish_date,langauge,like_tier,sentiment
0,Annoying him while making coffee 😂,Jayllnn,42,8,8,2023-10-26 00:00:00+00:00,UG,35273798,11898,,6aea6t0Jb_s,UCmX-8uO9dtnm4yQ5ldzcSdQ,,youtube#video,2023-09-20 02:22:46+00:00,en,a_little_viral,1
1,¿Qué son los DODS? #shorts,Pintos WHAT,50,0,0,2023-10-26 00:00:00+00:00,SV,11373294,2144,Redes y contacto: [https://zez.am/pintos_what]...,WrE3mgmC0Jo,UCcgrzE9nTEZX3yuMwmSWWPA,"pintos, what, team what, pintos what, parkour,...",youtube#video,2023-09-20 12:30:28+00:00,es,super_viral,1
2,Mert Demir - Ateşe Düştüm,Mert Demir,31,-5,19,2023-10-28 00:00:00+00:00,CY,10100369,2196,"Söz - Müzik: Mert Demir, Mela Bedel\nUd: Tarık...",RQmXet6kZ-Y,UCH9fW8ZWzymWpkz5Dljqcpw,,youtube#video,2023-09-21 00:00:00+00:00,tr,really_viral,0
3,Burna Boy - City Boys [Official Music Video],Burna Boy,42,0,8,2023-10-29 00:00:00+00:00,UG,18376708,7902,Burna Boy - City Boys\nStream/Download: https:...,hLDQ88vAhIs,UCEzDdNqNkT-7rSfSGSr1hWg,"burna boy, burner boy, burnaboy, Damini Ogulu,...",youtube#video,2023-09-22 00:00:00+00:00,en,super_viral,0
4,REMA NAMAKULA -TONYT,Rema Namakula,7,-2,43,2023-10-28 00:00:00+00:00,UG,1179277,2028,#uganda #remanamakula #tonyt #africanmusic #af...,Y3Q3YtyBMxE,UCmnuC2jgsyGso0hS5HPtCEw,"Rema, Ugandan Music, New Ugandan Music 2014, 2014",youtube#video,2023-09-22 00:00:00+00:00,,pretty_viral,0


In [5]:
df = pl.from_pandas(df) if not isinstance(df, pl.DataFrame) else df

train_df = df.filter(pl.col("publish_date") <= cutoff_date)
test_df = df.filter(pl.col("publish_date") > cutoff_date)


X_train = train_df.drop("like_tier")
X_test = test_df.drop("like_tier")

X_train_pd = X_train.to_pandas()    
X_test_pd = X_test.to_pandas()

y_train = train_df.get_column("like_tier")
y_test = test_df.get_column("like_tier")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

In [6]:
MAX_VOCAB = 5000
MAX_LEN = 50

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

# sample smaller size!
X_train_pd = X_train_pd.sample(100000, random_state=42)
y_train_mapped = y_train_mapped[X_train_pd.index]

# simple tokenization
def tokenize(text):
    return text.lower().split()

# build a vocabulary
counter = Counter()
for text in X_train_pd["text"]:
    counter.update(tokenize(text))

# keep top MAX_VOCAB words
# leave 0 for PAD, 1 for UNK
most_common = counter.most_common(MAX_VOCAB - 2)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
word2idx["<pad>"] = 0
word2idx["<unk>"] = 1

# convert to PyTorch Tensors
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

# mapping
def text_to_sequence(text):
    return [word2idx.get(word, 1) for word in tokenize(text)]

# add padding since RNN accepts equal length sequences
def pad_sequence(seq, max_len=MAX_LEN):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]

train_seq = [text_to_sequence(text) for text in X_train_pd["text"]]
test_seq = [text_to_sequence(text) for text in X_test_pd["text"]]

train_padded = torch.tensor([pad_sequence(seq) for seq in train_seq], dtype=torch.long).to(device)
test_padded = torch.tensor([pad_sequence(seq) for seq in test_seq], dtype=torch.long).to(device)

X_train_sentiment = torch.tensor(X_train_pd["sentiment"].to_numpy(), dtype=torch.float).unsqueeze(1).to(device)
X_test_sentiment = torch.tensor(X_test_pd["sentiment"].to_numpy(), dtype=torch.float).unsqueeze(1).to(device)

# Labels
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(train_padded, X_train_sentiment, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = TensorDataset(test_padded, X_test_sentiment, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=128)

print(train_padded.shape, test_padded.shape)

torch.Size([100000, 50]) torch.Size([90674, 50])


In [8]:
from sklearn.utils.class_weight import compute_class_weight

# hyperparams
EMBED_DIM = 64
HIDDEN_DIM = 32
OUTPUT_DIM = len(torch.unique(y_train_tensor))
EPOCHS = 10

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim + 1, output_dim)  # +1 for sentiment

    def forward(self, x, sentiment):  # <-- make sure sentiment is here!
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        last_hidden = hidden[-1]
        combined = torch.cat([last_hidden, sentiment], dim=1)
        return self.fc(combined)

model = SimpleRNN(vocab_size=MAX_VOCAB, embed_dim=EMBED_DIM, 
                  hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training RNN...")
for epoch in range(EPOCHS):
    total_loss = 0
    model.train()
    for X_batch, sentiment_batch, y_batch in train_loader:
        X_batch, sentiment_batch, y_batch = X_batch.to(device), sentiment_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch, sentiment_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, sentiment_batch, y_batch in test_loader:
        outputs = model(X_batch, sentiment_batch)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

print("\nRNN Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, all_preds, target_names=target_names, zero_division=0))

Training RNN...
Epoch 1, Loss: 1.0693
Epoch 2, Loss: 0.9521
Epoch 3, Loss: 0.9024
Epoch 4, Loss: 0.8690
Epoch 5, Loss: 0.8428
Epoch 6, Loss: 0.8191
Epoch 7, Loss: 0.7963
Epoch 8, Loss: 0.7745
Epoch 9, Loss: 0.7531
Epoch 10, Loss: 0.7326

RNN Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.89      0.81      0.85     73784
  pretty_viral       0.34      0.47      0.39     14234
  really_viral       0.20      0.29      0.23      2486
   super_viral       0.04      0.10      0.05       170

      accuracy                           0.74     90674
     macro avg       0.37      0.42      0.38     90674
  weighted avg       0.79      0.74      0.76     90674

